In [1]:
import pennylane as qml
from pennylane import numpy as np

dev = qml.device("default.qubit", wires=4)

@qml.qnode(dev)
def circuit(x, weights):
    qml.AngleEmbedding(x, wires=range(4))
    qml.StronglyEntanglingLayers(weights, wires=range(4))
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=4)
weights = np.random.random(size=shape)
print("weights shape:", shape)
print("output:", circuit(np.array([0.1, 0.2, 0.3, 0.4]), weights))
print(qml.draw(circuit)(np.array([0.1, 0.2, 0.3, 0.4]), weights))

weights shape: (2, 4, 3)
output: [tensor(0.36262683, requires_grad=True), tensor(-0.002691, requires_grad=True), tensor(0.35261063, requires_grad=True), tensor(0.36793604, requires_grad=True)]
0: ─╭AngleEmbedding(M0)─╭StronglyEntanglingLayers(M1)─┤  <Z>
1: ─├AngleEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Z>
2: ─├AngleEmbedding(M0)─├StronglyEntanglingLayers(M1)─┤  <Z>
3: ─╰AngleEmbedding(M0)─╰StronglyEntanglingLayers(M1)─┤  <Z>

M0 = 
[0.1 0.2 0.3 0.4]
M1 = 
[[[0.61398635 0.63356958 0.37864143]
  [0.93492575 0.46862733 0.6824113 ]
  [0.84633525 0.28033201 0.52314485]
  [0.40541884 0.63881099 0.7226106 ]]

 [[0.26655029 0.93215912 0.70760702]
  [0.38836133 0.78167528 0.64791503]
  [0.04992113 0.35743471 0.25109089]
  [0.02043296 0.95897774 0.0302823 ]]]


In [2]:
x = np.array([0.1, 0.2, 0.3, 0.4])
print(qml.draw(circuit, level="device", decimals=2)(x, weights))

0: ──RX(0.10)──Rot(0.61,0.63,0.38)─╭●───────╭X──Rot(0.27,0.93,0.71)─╭●────╭X────┤  <Z>
1: ──RX(0.20)──Rot(0.93,0.47,0.68)─╰X─╭●────│───Rot(0.39,0.78,0.65)─│──╭●─│──╭X─┤  <Z>
2: ──RX(0.30)──Rot(0.85,0.28,0.52)────╰X─╭●─│───Rot(0.05,0.36,0.25)─╰X─│──╰●─│──┤  <Z>
3: ──RX(0.40)──Rot(0.41,0.64,0.72)───────╰X─╰●──Rot(0.02,0.96,0.03)────╰X────╰●─┤  <Z>


In [3]:
import torch

@qml.qnode(dev, interface="torch")
def qcircuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(4))
    qml.StronglyEntanglingLayers(weights, wires=range(4))
    return [qml.expval(qml.PauliZ(i)) for i in range(4)]

qlayer = qml.qnn.TorchLayer(qcircuit, weight_shapes={"weights": (2, 4, 3)})

batch = torch.tensor([[0.1, 0.2, 0.3, 0.4],
                      [0.5, 0.6, 0.7, 0.8]])
out = qlayer(batch)
print(out)
print("output shape:", out.shape)
print("trainable params:", sum(p.numel() for p in qlayer.parameters()))

tensor([[-0.7166,  0.0966, -0.1906, -0.1096],
        [-0.6690,  0.1703, -0.2767, -0.2239]], grad_fn=<ViewBackward0>)
output shape: torch.Size([2, 4])
trainable params: 24


In [4]:
import math, wandb
import torch.nn as nn

class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Linear(1, 4)
        self.q = qml.qnn.TorchLayer(qcircuit, {"weights": (2, 4, 3)})
        self.decoder = nn.Linear(4, 1)

    def forward(self, x):
        return self.decoder(self.q(self.encoder(x)))

torch.manual_seed(42)
X = torch.linspace(-math.pi, math.pi, 100).unsqueeze(1)
y = torch.sin(X)

model = HybridModel()
opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
print("total params:", sum(p.numel() for p in model.parameters()))

run = wandb.init(project="datacenter-power-forecasting", name="pennylane-sine-warmup",
                 config={"qubits": 4, "layers": 2, "lr": 0.05, "epochs": 100})
for epoch in range(100):
    opt.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    opt.step()
    wandb.log({"loss": loss.item()})
    if epoch % 10 == 0 or epoch == 99:
        print(f"epoch {epoch:3d} | loss {loss.item():.5f}")
run.finish()

total params: 37


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/aryaneuse/.netrc.
wandb: Currently logged in as: aryan-faghirollahy (aryan-faghirollahy-uok) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


epoch   0 | loss 0.62931
epoch  10 | loss 0.05254
epoch  20 | loss 0.01153
epoch  30 | loss 0.00054
epoch  40 | loss 0.00082
epoch  50 | loss 0.00018
epoch  60 | loss 0.00007
epoch  70 | loss 0.00003
epoch  80 | loss 0.00003
epoch  90 | loss 0.00001
epoch  99 | loss 0.00000


loss,█▇▆▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss,0.0


In [5]:
from pathlib import Path
PROJECT_DIR = Path.cwd().parent
path = PROJECT_DIR / "docs" / "literature.md"

papers = [
    ("The MIT Supercloud Dataset", "https://arxiv.org/abs/2108.02037"),
    ("Short-Term Load Forecasting for AI-Data Center", "https://arxiv.org/abs/2503.07756"),
    ("Electricity Demand and Grid Impacts of AI Data Centers", "https://arxiv.org/abs/2509.07218"),
    ("Better than classical? The subtle art of benchmarking QML (Bowles et al.)", "https://arxiv.org/abs/2403.07059"),
    ("Quantum neural networks for power flow analysis (Kaseb et al., EPSR 2024)", "https://doi.org/10.1016/j.epsr.2024.110677"),
    ("Quantum Reservoir Computing for STLF (optional)", "https://arxiv.org/abs/2606.12806"),
]

text = "# Literature Notes\n\n"
for i, (title, url) in enumerate(papers, 1):
    text += (f"## {i}. {title}\n\n**Link:** {url}\n\n"
             "**Idea:**\n\n**Method:**\n\n**Results:**\n\n**Weaknesses / Notes for this project:**\n\n---\n\n")

if path.exists():
    print("already exists, not overwritten:", path)
else:
    path.write_text(text, encoding="utf-8")
    print("created:", path)

created: /Users/aryaneuse/Desktop/datacenter-power-forecasting/docs/literature.md


In [7]:
import time, urllib.request
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
PAPERS_DIR = PROJECT_DIR / "docs" / "papers"
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

ARXIV = {
    # core (week 1 reading)
    "01_mit_supercloud_dataset":        "2108.02037",
    "02_alberta_ai_datacenter_stlf":    "2503.07756",
    "03_ai_datacenter_grid_impacts":    "2509.07218",
    "04_bowles_benchmarking_qml":       "2403.07059",
    "05_qrc_load_forecasting":          "2606.12806",
    "06_physics_aware_gpu_power":       "2605.04074",
    # quantum ML for energy
    "10_wiser_eon_hybrid_qml":          "2605.24252",
    "11_pv_forecasting_qml":            "2312.16379",
    "12_quantum_smart_grid_review":     "2609.18847",
    "13_qaoa_load_scheduling":          "2311.10594",
    "14_hybrid_qnn_review":             "2608.01194",
    "15_qlstm":                         "2009.01783",
    # grid, control, uncertainty
    "20_choukse_power_stabilization":   "2508.14318",
    "21_ko_hybrid_ess_supercap":        "2512.08076",
    "22_differentiable_predictive_hess":"2606.04869",
    "23_adaptive_conformal_inference":  "2106.00170",
    "24_foundation_model_leakage":      "2510.13654",
}

for name, aid in ARXIV.items():
    out = PAPERS_DIR / f"{name}_{aid}.pdf"
    if out.exists():
        print(f"skip  {out.name}")
        continue
    try:
        req = urllib.request.Request(f"https://arxiv.org/pdf/{aid}",
                                     headers={"User-Agent": "Mozilla/5.0 (literature archive)"})
        out.write_bytes(urllib.request.urlopen(req, timeout=60).read())
        print(f"ok    {out.name}  ({out.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"FAIL  {name} ({aid}): {e}")
    time.sleep(3)   # arXiv asks for a pause between requests

print(f"\n{len(list(PAPERS_DIR.glob('*.pdf')))} PDFs in {PAPERS_DIR}")

skip  01_mit_supercloud_dataset_2108.02037.pdf
skip  02_alberta_ai_datacenter_stlf_2503.07756.pdf
skip  03_ai_datacenter_grid_impacts_2509.07218.pdf
skip  04_bowles_benchmarking_qml_2403.07059.pdf
skip  05_qrc_load_forecasting_2606.12806.pdf
skip  06_physics_aware_gpu_power_2605.04074.pdf
skip  10_wiser_eon_hybrid_qml_2605.24252.pdf
FAIL  11_pv_forecasting_qml (2312.16379): IncompleteRead(8388608 bytes read, 10615021 more expected)
skip  12_quantum_smart_grid_review_2609.18847.pdf
skip  13_qaoa_load_scheduling_2311.10594.pdf
skip  14_hybrid_qnn_review_2608.01194.pdf
skip  15_qlstm_2009.01783.pdf
skip  20_choukse_power_stabilization_2508.14318.pdf
skip  21_ko_hybrid_ess_supercap_2512.08076.pdf
skip  22_differentiable_predictive_hess_2606.04869.pdf
skip  23_adaptive_conformal_inference_2106.00170.pdf
skip  24_foundation_model_leakage_2510.13654.pdf

16 PDFs in /Users/aryaneuse/Desktop/datacenter-power-forecasting/docs/papers


In [9]:
import time, urllib.request

OPEN_ACCESS = {
    "30_sci_reports_2025_quantum_load":  "https://www.nature.com/articles/s41598-025-89933-x.pdf",
    "31_qsvm_household_jcc_2024":        "https://link.springer.com/content/pdf/10.1186/s13677-024-00669-x.pdf",
    "32_cerezo_barren_plateaus_2025":    "https://www.nature.com/articles/s41467-025-63099-6.pdf",
    "33_qsegrnn_transformer_temp_2025":  "https://link.springer.com/content/pdf/10.1140/epjqt/s40507-025-00333-6.pdf",
    "34_nerc_emerging_large_loads_2025": "https://www.nerc.com/globalassets/who-we-are/standing-committees/rstc/whitepaper-characteristics-and-risks-of-emerging-large-loads.pdf",
    "35_epri_powering_intelligence_2024":"https://restservice.epri.com/publicdownload/000000003002028905/0/Product",
}

for name, url in OPEN_ACCESS.items():
    out = PAPERS_DIR / f"{name}.pdf"
    if out.exists():
        print(f"skip    {out.name}")
        continue
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (literature archive)"})
        data = urllib.request.urlopen(req, timeout=60).read()
        if data[:5] != b"%PDF-":
            print(f"NOT PDF {name} (probably a login or landing page)")
            continue
        out.write_bytes(data)
        print(f"ok      {out.name}  ({len(data)/1e6:.1f} MB)")
    except Exception as e:
        print(f"FAIL    {name}: {e}")
    time.sleep(2)

print(f"\n{len(list(PAPERS_DIR.glob('*.pdf')))} PDFs in {PAPERS_DIR}")

ok      30_sci_reports_2025_quantum_load.pdf  (7.5 MB)
ok      31_qsvm_household_jcc_2024.pdf  (2.1 MB)
ok      32_cerezo_barren_plateaus_2025.pdf  (1.3 MB)
ok      33_qsegrnn_transformer_temp_2025.pdf  (3.1 MB)
ok      34_nerc_emerging_large_loads_2025.pdf  (4.2 MB)
FAIL    35_epri_powering_intelligence_2024: Remote end closed connection without response

22 PDFs in /Users/aryaneuse/Desktop/datacenter-power-forecasting/docs/papers


In [13]:
from pathlib import Path

PAPERS_DIR = Path.cwd().parent / "docs" / "papers"

EXPECTED = [
    "01_mit_supercloud_dataset_2108.02037.pdf",
    "02_alberta_ai_datacenter_stlf_2503.07756.pdf",
    "03_ai_datacenter_grid_impacts_2509.07218.pdf",
    "04_bowles_benchmarking_qml_2403.07059.pdf",
    "05_qrc_load_forecasting_2606.12806.pdf",
    "06_physics_aware_gpu_power_2605.04074.pdf",
    "10_wiser_eon_hybrid_qml_2605.24252.pdf",
    "11_pv_forecasting_qml_2312.16379.pdf",
    "12_quantum_smart_grid_review_2609.18847.pdf",
    "13_qaoa_load_scheduling_2311.10594.pdf",
    "14_hybrid_qnn_review_2608.01194.pdf",
    "15_qlstm_2009.01783.pdf",
    "20_choukse_power_stabilization_2508.14318.pdf",
    "21_ko_hybrid_ess_supercap_2512.08076.pdf",
    "22_differentiable_predictive_hess_2606.04869.pdf",
    "23_adaptive_conformal_inference_2106.00170.pdf",
    "24_foundation_model_leakage_2510.13654.pdf",
    "30_sci_reports_2025_quantum_load.pdf",
    "31_qsvm_household_jcc_2024.pdf",
    "32_cerezo_barren_plateaus_2025.pdf",
    "33_qsegrnn_transformer_temp_2025.pdf",
    "34_nerc_emerging_large_loads_2025.pdf",
    "35_epri_powering_intelligence_2024.pdf",
    "36_kaseb_phd_thesis_2026.pdf",
    "37_iea_energy_and_ai_2025.pdf",
    "38_vqc_datacenter_energy_control_2024.pdf",
]
PENDING = [  # to download at university
    "39_colangelo_nature_energy_2026.pdf",
    "40_qmi_forecast_schedule_2026.pdf",
]

def check(path):
    if not path.exists():
        return "MISSING", 0
    data = path.read_bytes()
    size = len(data) / 1e6
    if data[:5] != b"%PDF-":
        return "NOT PDF", size
    if b"%%EOF" not in data[-2048:]:
        return "TRUNCATED", size
    return "ok", size

problems = 0
for name in EXPECTED + PENDING:
    status, size = check(PAPERS_DIR / name)
    if name in PENDING and status == "MISSING":
        status = "PENDING"
    elif status != "ok":
        problems += 1
    print(f"{status:10s} {size:5.1f} MB  {name}")

extra = sorted(p.name for p in PAPERS_DIR.glob("*.pdf") if p.name not in EXPECTED + PENDING)
print("\nUnexpected files:", extra if extra else "none")
print(f"Summary: {len(EXPECTED) - problems}/{len(EXPECTED)} ok, "
      f"{sum((PAPERS_DIR / n).exists() for n in PENDING)}/{len(PENDING)} university papers present")

ok           0.7 MB  01_mit_supercloud_dataset_2108.02037.pdf
ok           7.0 MB  02_alberta_ai_datacenter_stlf_2503.07756.pdf
ok           6.8 MB  03_ai_datacenter_grid_impacts_2509.07218.pdf
ok           5.8 MB  04_bowles_benchmarking_qml_2403.07059.pdf
ok           6.5 MB  05_qrc_load_forecasting_2606.12806.pdf
ok           1.3 MB  06_physics_aware_gpu_power_2605.04074.pdf
ok           2.2 MB  10_wiser_eon_hybrid_qml_2605.24252.pdf
ok          19.0 MB  11_pv_forecasting_qml_2312.16379.pdf
ok           0.7 MB  12_quantum_smart_grid_review_2609.18847.pdf
ok           3.1 MB  13_qaoa_load_scheduling_2311.10594.pdf
ok           1.6 MB  14_hybrid_qnn_review_2608.01194.pdf
ok           1.8 MB  15_qlstm_2009.01783.pdf
ok           1.1 MB  20_choukse_power_stabilization_2508.14318.pdf
ok           1.2 MB  21_ko_hybrid_ess_supercap_2512.08076.pdf
ok           2.9 MB  22_differentiable_predictive_hess_2606.04869.pdf
ok           1.2 MB  23_adaptive_conformal_inference_2106.00170.pdf
ok      

In [14]:
!aws s3 ls --no-sign-request s3://mit-supercloud-dataset/datacenter-challenge/

                           PRE 202201/


In [15]:
!aws s3 ls --no-sign-request s3://mit-supercloud-dataset/datacenter-challenge/202201/

                           PRE cpu/
                           PRE gpu/
2022-01-19 19:11:40        285 LICENSE
2022-01-30 00:45:11       7616 README.md
2022-01-20 17:20:09        339 labelled_job_stats.csv
2022-01-20 17:20:10      82332 labelled_jobids.csv
2022-01-20 17:20:10 2274303873 node-data.csv
2022-01-20 17:20:11  103165427 slurm-log.csv
2022-01-20 17:20:11        111 tres-mapping.txt


In [16]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
readme = PROJECT_DIR / "README.md"

section = """
## Data source

MIT Supercloud Datacenter Challenge dataset (CC BY-NC-ND 4.0), public AWS S3 bucket, no account needed.

- Prefix: `s3://mit-supercloud-dataset/datacenter-challenge/202201/`
- List: `aws s3 ls --no-sign-request s3://mit-supercloud-dataset/datacenter-challenge/202201/`
- Used: `gpu/` (one CSV per job, 100 ms), `slurm-log.csv` (~103 MB), `labelled_jobids.csv`
- Not used: `cpu/`, `node-data.csv` (~2.27 GB)
- Raw and processed data are never committed (license: NoDerivatives); only scripts are published.
"""

with open(readme, "a", encoding="utf-8") as f:
    f.write(section)

print("README.md updated:", readme)
print(readme.read_text(encoding="utf-8")[-600:])

README.md updated: /Users/aryaneuse/Desktop/datacenter-power-forecasting/README.md

## Data source

MIT Supercloud Datacenter Challenge dataset (CC BY-NC-ND 4.0), public AWS S3 bucket, no account needed.

- Prefix: `s3://mit-supercloud-dataset/datacenter-challenge/202201/`
- List: `aws s3 ls --no-sign-request s3://mit-supercloud-dataset/datacenter-challenge/202201/`
- Used: `gpu/` (one CSV per job, 100 ms), `slurm-log.csv` (~103 MB), `labelled_jobids.csv`
- Not used: `cpu/`, `node-data.csv` (~2.27 GB)
- Raw and processed data are never committed (license: NoDerivatives); only scripts are published.



In [17]:
import urllib.request
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd().parent
out = PROJECT_DIR / "data" / "raw" / "tsfm_lineage.csv"   # data/ is gitignored
out.parent.mkdir(parents=True, exist_ok=True)

url = ("https://github.com/DAG-UPB/rethinking-tsfm-evaluation/raw/refs/heads/main/"
       "lineage-analysis/Time-Series-Foundation-Models-Lineage.csv")
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=60) as r:
    out.write_bytes(r.read())
print("downloaded:", out.stat().st_size, "bytes")

df = pd.read_csv(out)
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.head(3).to_string(max_colwidth=40))

# Keyword search across all columns (case-insensitive)
keywords = ["supercloud", "mit ", "gpu", "hpc", "datacenter", "data center",
            "cluster", "slurm", "marconi", "pm100", "power", "borg", "alibaba"]
text = df.astype(str).apply(lambda col: col.str.lower())
print("\n--- keyword hits ---")
for kw in keywords:
    mask = text.apply(lambda col: col.str.contains(kw, regex=False)).any(axis=1)
    print(f"{kw!r}: {int(mask.sum())} row(s)")
    if mask.any():
        print(df[mask].to_string(max_colwidth=40)[:1500])

downloaded: 38434 bytes
shape: (401, 34)
columns: ['Dataset', 'Source', 'Domain', 'Frequency', 'Chronos', 'TimesFM', 'Moirai', 'Moirai-MoE', 'Time-GPT', 'ForecastPFN', 'MOMENT', 'Lag-Llama', 'GPT4TS', 'Timer', 'VisionTS', 'TTM', 'LLMTime', 'Time-LMM', 'UniTime', 'Time-MoE', 'Chronos-2', 'Flowstate', 'LightGTS', 'Sundial', 'TabPFN-TS', 'Time-MoE2', 'TinyTimeMixer', 'TiRex', 'Toto', 'YingLong (under review)', 'Gift-Eval', 'Flowstate(T)', 'TimesFM-ICL', 'ROSE']
        Dataset                        Source  Domain Frequency Chronos TimesFM Moirai Moirai-MoE  Time-GPT ForecastPFN MOMENT Lag-Llama GPT4TS Timer VisionTS  TTM LLMTime Time-LMM UniTime Time-MoE Chronos-2 Flowstate LightGTS Sundial  TabPFN-TS  Time-MoE2  TinyTimeMixer TiRex Toto  YingLong (under review) Gift-Eval Flowstate(T) TimesFM-ICL ROSE
0  Jena Weather  Autoformer (Wu et al., 2021)  Nature       10T     NaN     NaN     ZS         ZS       NaN         NaN    T/T       NaN    T/T    ZS       ZS   ZS      ZS      NaN     NaN 

In [18]:
cols = ["Chronos", "TimesFM", "Toto", "Moirai", "TiRex", "Chronos-2"]
print(df.loc[df["Dataset"].isin(["Alibaba Cluster Trace 2018", "Borg Cluster Data 2011"]),
             ["Dataset"] + cols].to_string())

                       Dataset Chronos TimesFM Toto Moirai TiRex Chronos-2
19  Alibaba Cluster Trace 2018     NaN     NaN    P      P   NaN       NaN
21      Borg Cluster Data 2011     NaN     NaN    P      P     P       NaN


In [19]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
lit = PROJECT_DIR / "docs" / "literature.md"
text = lit.read_text(encoding="utf-8")

paragraph = """**Lineage check (1 Oct 2026).** We downloaded the paper's lineage table (401 datasets, 30 model columns) and searched all columns for supercloud, mit, gpu, hpc, datacenter, slurm, marconi and pm100: no hits. The only cluster-workload datasets listed are Alibaba Cluster Trace 2018 and Borg Cluster Data 2011 (CloudOps, 5-minute frequency). Both are marked as pre-training data for Toto and Moirai (Borg also for TiRex), and for none of Chronos, TimesFM or Chronos-2. They are CPU/memory-type traces from other systems and periods, with no temporal overlap with our 2021 data, but Toto has seen cluster-type workloads, so its zero-shot results on our series are not from an entirely unseen domain. This is evidence of absence from documented public corpora only; proprietary pre-training data (e.g., for Toto) cannot be checked."""

if "**Lineage check" in text:
    print("Already present, nothing changed.")
else:
    marker = "\n---\n\n## 9."
    idx = text.find(marker)
    if idx == -1:
        raise SystemExit("Marker '## 9.' not found; check literature.md structure.")
    new_text = text[:idx].rstrip("\n") + "\n\n" + paragraph + "\n" + text[idx:]
    lit.write_text(new_text, encoding="utf-8")
    print("literature.md updated.\n")
    start = max(0, idx - 400)
    print(new_text[start: idx + len(paragraph) + 80])

literature.md updated.

ing sources (to be verified from the lineage CSV and model cards). Our test period (autumn 2021) precedes every model's pre-training cutoff, so Requirement 2 cannot be met; this is stated as a limitation. Models we train ourselves use a strict chronological split with train-only normalization, which this paper does not call into question. PM100 is used as a second dataset to check rank stability.

**Lineage check (1 Oct 2026).** We downloaded the paper's lineage table (401 datasets, 30 model columns) and searched all columns for supercloud, mit, gpu, hpc, datacenter, slurm, marconi and pm100: no hits. The only cluster-workload datasets listed are Alibaba Cluster Trace 2018 and Borg Cluster Data 2011 (CloudOps, 5-minute frequency). Both are marked as pre-training data for Toto and Moirai (Borg also for TiRex), and for none of Chronos, TimesFM or Chronos-2. They are CPU/memory-type traces from other systems and periods, with no temporal overlap with our 2021 data,